# BD Vectorielle & Embeddings — Classification d'articles
**Auteurs :** Van Zieleghem, Dai Pra  
**Cours :** Intelligence Artificielle  
**Date :** 2025-2026

---

## Objectif
Lire un article au format Word (`.docx`), le tokeniser, générer son embedding, puis interroger une base de données vectorielle **FAISS** pour déterminer si l'article est **de sport** ou **de cuisine**.

## Workflow suivi
```
Texte (.docx) → Tokenisation → Embedding → Index FAISS → Recherche → Classification
```

## Table des matières
1. [Installation des dépendances](#installation)
2. [Lecture et tokenisation du document Word](#lecture)
3. [Embeddings — vectorisation des articles](#embeddings)
4. [Base de données FAISS — indexation](#faiss)
5. [Classification d'un article inconnu](#classification)
6. [Visualisation PCA](#pca)
7. [Sauvegarde de l'index](#sauvegarde)
8. [Tests et résultats](#tests)

## 1. Installation des dépendances <a id='installation'></a>

In [ ]:
# Installer sentence-transformers
import sys
!{sys.executable} -m pip install sentence-transformers

In [ ]:
# Installer FAISS
import sys
!{sys.executable} -m pip install faiss-cpu

In [ ]:
# Installer python-docx pour lire les fichiers Word
import sys
!{sys.executable} -m pip install python-docx

## 2. Lecture et tokenisation du document Word <a id='lecture'></a>

On extrait le texte brut du fichier `.docx`, puis on le tokenise en ignorant la ponctuation.  

> **Utilisation de l'IA :** La fonction de tokenisation (expression régulière pour supprimer la ponctuation tout en conservant les accents) a été élaborée avec l'aide de Claude (Anthropic). Voir le mini rapport en fin de notebook.

In [ ]:
from docx import Document
import re

def lire_docx(chemin):
    """
    Lit un fichier Word (.docx) et retourne son contenu textuel complet.
    Les paragraphes vides sont ignorés.

    Paramètre :
        chemin (str) : chemin vers le fichier .docx

    Retourne :
        str : texte complet du document, paragraphes séparés par des espaces
    """
    doc = Document(chemin)
    paragraphes = [para.text for para in doc.paragraphs if para.text.strip()]
    return " ".join(paragraphes)

def tokeniser(texte):
    """
    Tokenise un texte en ignorant la ponctuation et les tokens mixtes (ex: '1er', '90km').
    Approche suggérée par IA puis améliorée : re.findall avec lettres accentuées uniquement.
    On ne garde que les séquences de lettres pures (a-z + accents français).
    Avantage : évite les tokens mixtes comme '90km' ou '1er' qui passaient avec la regex précédente.
    """
    tokens = re.findall(r'[a-zA-ZÀ-ÿ]+', texte.lower())
    return tokens

# --- Test rapide ---
test = "Le joueur a marqué 3 buts! C'est incroyable... Bravo à lui."
print("Texte original :", test)
print("Tokens         :", tokeniser(test))
assert "incroyable" in tokeniser(test), "ERREUR : mot accentué absent"
assert "3" not in tokeniser(test), "ERREUR : chiffre isolé présent"
assert "buts" in tokeniser(test), "ERREUR : mot simple absent"
print("Assertions OK — tokenisation validée.")

Test unitaire avec assertions : la ponctuation (`!`, `...`, `'`), les nombres purs et les tokens mixtes sont supprimés. Les mots accentués sont conservés. Les trois assertions passent sans erreur.

## 3. Embeddings — vectorisation des articles <a id='embeddings'></a>

On utilise le modèle `all-MiniLM-L6-v2` de `sentence-transformers` pour transformer chaque article en vecteur numérique (embedding).  
Ce modèle produit des vecteurs de **384 dimensions**.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Chargement du modèle
model = SentenceTransformer('all-MiniLM-L6-v2')

# Lecture et tokenisation des articles de référence
texte_sport   = lire_docx("article_sport.docx")
texte_cuisine = lire_docx("article_cuisine.docx")

# Les phrases/documents à indexer
sentences = [texte_sport, texte_cuisine]
labels    = ["sport", "cuisine"]

# Vectoriser les phrases
embeddings = model.encode(sentences)

# Afficher les dimensions de la matrice embeddings
print("Dimensions de la matrice embeddings :", embeddings.shape)
print(f"→ {embeddings.shape[0]} articles, {embeddings.shape[1]} dimensions par vecteur")

> **Note :** Le texte brut est passé directement au modèle car `all-MiniLM-L6-v2` est un transformer entraîné sur du texte naturel avec ponctuation et casse. Passer un texte dégradé (sans ponctuation, tout en minuscules) réduit la qualité sémantique de l'embedding. La tokenisation reste définie et testée en section 2 pour l'analyse linguistique, mais elle n'est pas appliquée avant l'encoding.

## 4. Base de données FAISS — indexation <a id='faiss'></a>

On crée un index FAISS (`IndexFlatL2`) qui utilise la **distance euclidienne (L2)** pour mesurer la similarité entre vecteurs.  
Plus la distance est petite, plus les articles sont similaires.

In [ ]:
import faiss
import numpy as np

# 1. Créer un index FAISS (distance L2, dimension 384)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

# 2. Ajouter les embeddings à l'index (FAISS exige float32)
index.add(embeddings.astype(np.float32))

print(f"Index FAISS créé : {index.ntotal} vecteur(s) indexé(s)")

## 5. Classification d'un article inconnu <a id='classification'></a>

On lit l'article à classifier, on génère son embedding, puis on interroge FAISS.  
L'article dont la distance L2 est la plus **petite** détermine la catégorie prédite.

In [ ]:
def classifier_article(chemin_article):
    """
    Classifie un article Word en 'sport' ou 'cuisine' via recherche FAISS.

    Paramètre :
        chemin_article (str) : chemin vers le fichier .docx à classifier

    Retourne :
        str : catégorie prédite ('sport' ou 'cuisine')
    """
    
    # Lecture et tokenisation de l'article à tester
    texte = lire_docx(chemin_article)
    
    # Génération de l'embedding
    query_embedding = model.encode([texte]).astype(np.float32)
    
    # 3. Rechercher des embeddings similaires (k=2 pour voir les 2 résultats)
    k = 2
    distances, indices_result = index.search(query_embedding, k)
    
    print(f"\nArticle testé : {chemin_article}")
    print("Résultats les plus proches :")
    for i, distance in zip(indices_result[0], distances[0]):
        print(f"  - '{labels[i]}' (distance L2 : {distance:.4f})")
    
    categorie_predite = labels[indices_result[0][0]]
    print(f">>> CLASSIFICATION : '{categorie_predite.upper()}' <<<")
    return categorie_predite

## 6. Visualisation PCA <a id='pca'></a>

On réduit les embeddings à 2 dimensions avec une PCA pour visualiser la proximité entre les articles.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

plt.figure(figsize=(8, 5))
colors = ["blue", "red"]
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=colors)
for i, label in enumerate(labels):
    plt.annotate(label, (embeddings_2d[i, 0], embeddings_2d[i, 1]), fontsize=12)
plt.title("Visualisation PCA des embeddings (articles de référence)")
plt.xlabel("Composante 1")
plt.ylabel("Composante 2")
plt.show()

## 7. Sauvegarde de l'index <a id='sauvegarde'></a>

On sauvegarde l'index FAISS sur disque pour pouvoir le réutiliser sans recalculer les embeddings.

In [ ]:
import os

faiss.write_index(index, "mon_index.faiss")
# Pour recharger : index2 = faiss.read_index("mon_index.faiss")

repertoire_courant = os.getcwd()
print(f"Index sauvegardé dans : {repertoire_courant}/mon_index.faiss")

## 8. Tests et résultats <a id='tests'></a>

On teste sur 3 fichiers :
- `article_sport.docx` → attendu : **sport**
- `article_cuisine.docx` → attendu : **cuisine**  
- `article_test.docx` → article ambigu (Tour de France + nutrition)

In [ ]:
print("=" * 50)
print("TEST 1 — Article de sport")
print("=" * 50)
r1 = classifier_article("article_sport.docx")

print()
print("=" * 50)
print("TEST 2 — Article de cuisine")
print("=" * 50)
r2 = classifier_article("article_cuisine.docx")

print()
print("=" * 50)
print("TEST 3 — Article ambigu (sport + nutrition)")
print("=" * 50)
r3 = classifier_article("article_test.docx")

In [ ]:
# Récapitulatif
print("\n" + "=" * 55)
print("RÉCAPITULATIF")
print("=" * 55)
print(f"{'Fichier':<28} {'Attendu':<12} {'Prédit':<12} {'OK?'}")
print("-" * 55)
tests = [
    ("article_sport.docx",   "sport",   r1),
    ("article_cuisine.docx", "cuisine", r2),
    ("article_test.docx",    "?",       r3),
]
for fichier, attendu, predit in tests:
    ok = "✓" if (attendu == predit or attendu == "?") else "✗"
    print(f"{fichier:<28} {attendu:<12} {predit:<12} {ok}")

---

## Mini rapport

### Utilisation de l'IA générative

L'IA (Claude, Anthropic) a été utilisée à deux niveaux :

**1. Tokenisation sans ponctuation**  
La consigne demandait explicitement d'utiliser une IA pour cette partie. Nous avons demandé à Claude de proposer une fonction Python de tokenisation ignorant la ponctuation tout en conservant les caractères accentués. L'IA a d'abord suggéré `re.sub(r"[^\w\s]", " ", texte, flags=re.UNICODE)` combinée à un filtre `not t.isdigit()`. Nous avons identifié une limite de cette approche : les tokens mixtes comme `'90km'` ou `'1er'` passaient sans être filtrés. Nous avons donc amélioré la solution en adoptant `re.findall(r'[a-zA-ZÀ-ÿ]+', texte.lower())`, qui ne conserve que les séquences de lettres pures et résout ce problème. Cette itération illustre qu'il faut toujours tester et comprendre le code suggéré par l'IA avant de l'intégrer.

**2. Génération des fichiers Word de test**  
Les trois articles `.docx` utilisés comme données de test ont été générés par Claude, avec des contenus réalistes pour les catégories sport et cuisine, ainsi qu'un article volontairement ambigu (Tour de France + nutrition).

---

### Analyse réflexive

**Points positifs :**  
L'IA a permis de gagner du temps sur la tokenisation (gestion Unicode, regex) sans valeur ajoutée pédagogique directe. Elle a produit des données de test cohérentes et variées rapidement.

**Points d'attention :**  
La suggestion initiale de l'IA présentait une limite non évidente : `\w` inclut chiffres et underscores, laissant passer des tokens mixtes comme `'90km'`. Il a fallu comprendre ce comportement, le tester, et proposer une regex alternative plus stricte. L'IA ne remplace pas le jugement technique.

**Limites du système :**  
Avec seulement 2 articles de référence dans FAISS, le système est fragile. En production, on alimenterait la base avec des dizaines d'articles par catégorie.

---

### Organisation des tests

| Test | Fichier | Attendu | Rôle |
|------|---------|---------|------|
| Unitaire tokenisation | (inline) | tokens sans ponctuation | Valide `tokeniser()` |
| Classification sport | `article_sport.docx` | `sport` | Valide la détection sport |
| Classification cuisine | `article_cuisine.docx` | `cuisine` | Valide la détection cuisine |
| Cas limite | `article_test.docx` | `?` | Explore le comportement ambigu |